In [1]:
!pip uninstall -y tensorflow keras
!pip install tensorflow==2.15.0

Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0
Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
  Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (475.3 MB)
Using cached keras-2.15.0-py3-none-any.whl (1.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.15.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.15.0 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from keras.models import Sequential, load_model
from keras.layers import Dense

data_basepath = '/content/sample_data/'
# data_basepath = '..\\data\\spain_dataframes\\'
model_basepath = '/content/sample_data/'
df = pd.read_csv(f'{data_basepath}cleaned_data.csv')

In [4]:
# Cargar datos
df = pd.read_csv(f"{data_basepath}cleaned_data.csv")

mean_df_by_comunidad = df.groupby("comunidad_autonoma")[["dias_alta_contaminacion", "percepcion_seguridad"]].mean()

# Guarda las medias para posteriores predicciones
mean_df_by_comunidad.to_csv(f"{data_basepath}mean_df_by_comunidad.csv")

# Limpieza de columnas innecesarias para entrenar el modelo (variables no rellenables por el usuario)
# df = df.drop(["dias_alta_contaminacion", "percepcion_seguridad", "frecuencia_felicidad"], axis=1)
df = df.drop("frecuencia_felicidad", axis=1)
df.head()

,comunidad_autonoma,edad,genero,actividad_fisica,asistencia_cine,asistencia_directos,asistencia_cultural,asistencia_deporte,dias_alta_contaminacion,estudios,estado_civil,horasTrabajadas_mes,salario_anual,satisf_hospitales,satisf_dentistas,satisf_especialistas,satisf_medGeneral,percepcion_seguridad,nivel_felicidad
0,Extremadura,10,hombre,nivel_alto,no,no_puede,no,no,3.3774,primaria,soltero/a,0.0000,0.0000,satisfecho/a,muy_satisfecho/a,insatisfecho/a,satisfecho/a,8.260975,8.1
1,Extremadura,18,mujer,nivel_alto,no,no,no_puede,no,1.4413,educacion_superior,casado/a,127.4882,23284.5483,insatisfecho/a,satisfecho/a,satisfecho/a,satisfecho/a,8.376973,6.1
2,Extremadura,41,hombre,nivel_bajo,no,si,si,no,6.7861,primaria,soltero/a,153.6451,21607.6075,satisfecho/a,muy_satisfecho/a,satisfecho/a,satisfecho/a,8.912742,8.0
3,Extremadura,17,mujer,nivel_alto,si,no,no,no,2.5476,primero_secundaria,soltero/a,132.2961,29640.1647,insatisfecho/a,neutral,satisfecho/a,satisfecho/a,9.091575,8.4
4,Extremadura,75,hombre,nivel_bajo,no,no,si,si,4.7876,segundo_secundaria_general,soltero/a,131.8787,17538.1113,satisfecho/a,satisfecho/a,insatisfecho/a,satisfecho/a,7.887052,8.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   comunidad_autonoma       20000 non-null  object 
 1   edad                     20000 non-null  int64  
 2   genero                   19370 non-null  object 
 3   actividad_fisica         20000 non-null  object 
 4   asistencia_cine          20000 non-null  object 
 5   asistencia_directos      20000 non-null  object 
 6   asistencia_cultural      20000 non-null  object 
 7   asistencia_deporte       20000 non-null  object 
 8   dias_alta_contaminacion  20000 non-null  float64
 9   estudios                 20000 non-null  object 
 10  estado_civil             20000 non-null  object 
 11  horasTrabajadas_mes      20000 non-null  float64
 12  salario_anual            20000 non-null  float64
 13  satisf_hospitales        20000 non-null  object 
 14  satisf_dentistas      

In [5]:
# Separar características y variable objetivo
X = df.drop("nivel_felicidad", axis=1)
y = df["nivel_felicidad"]

# Detectar columnas numéricas y categóricas
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Crear transformador para preprocesamiento
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# Dividir en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Aplicar preprocesamiento a X_train y X_test
X_train_prep = preprocessor.fit_transform(X_train).toarray()
X_test_prep = preprocessor.transform(X_test).toarray()

# Ajustar input_shape al modelo
input_dim = X_train_prep.shape[1]

# Crear pipeline de preprocesamiento y entrenamiento
model = Sequential([
    Dense(64, activation="relu", input_dim=input_dim),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# Entrenar el modelo
# early_stop = EarlyStopping(patience=10, restore_best_weights=True)
# model.fit(X_train_prep, y_train, epochs=100, batch_size=32,
#           validation_split=0.2, callbacks=[early_stop], verbose=1)
model.fit(X_train_prep, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Evaluar
loss, mae = model.evaluate(X_test_prep, y_test)
print(f"MAE en test: {mae:.2f}")

Epoch 1/50
400/400 [==============================] - 3s 5ms/step - loss: 3.6934 - mae: 1.3250 - val_loss: 1.0856 - val_mae: 0.9459
Epoch 2/50
400/400 [==============================] - 1s 3ms/step - loss: 1.1172 - mae: 0.9244 - val_loss: 1.0652 - val_mae: 0.8984
Epoch 3/50
400/400 [==============================] - 1s 2ms/step - loss: 1.1001 - mae: 0.9195 - val_loss: 1.0802 - val_mae: 0.9269
Epoch 4/50
400/400 [==============================] - 1s 2ms/step - loss: 1.0940 - mae: 0.9177 - val_loss: 1.0793 - val_mae: 0.9227
Epoch 5/50
400/400 [==============================] - 1s 3ms/step - loss: 1.0821 - mae: 0.9121 - val_loss: 1.1240 - val_mae: 0.8309
Epoch 6/50
400/400 [==============================] - 1s 3ms/step - loss: 1.0816 - mae: 0.9084 - val_loss: 1.0769 - val_mae: 0.9076
Epoch 7/50
400/400 [==============================] - 1s 3ms/step - loss: 1.0773 - mae: 0.9029 - val_loss: 1.0903 - val_mae: 0.8994
Epoch 8/50
400/400 [==============================] - 1s 2ms/step - loss: 1.

In [7]:
# Guardar modelo como .keras
model.save(f"{model_basepath}modelo_felicidad.keras")
model.save(f"{model_basepath}models/modelo_felicidad.keras")
print("Modelo guardado como modelo_felicidad.keras")

Modelo guardado como modelo_felicidad.keras


In [8]:
# Guardar el preprocessor como .pkl
joblib.dump(preprocessor, f"{model_basepath}preprocessor.pkl")
print("Preprocessor guardado como preprocessor.pkl")

Preprocessor guardado como preprocessor.pkl


In [ ]:
# Prueba de carga del modelo
model_loaded = load_model(f"{model_basepath}modelo_felicidad.keras")
print("Modelo cargado correctamente.")

Modelo cargado correctamente.


In [ ]:
# Prueba de carga del preprocessor
preprocessor_loaded = joblib.load(f"{model_basepath}preprocessor.pkl")
print("Preprocessor cargado correctamente.")

Preprocessor cargado correctamente.


In [ ]:
# Prueba de predicción
data_prueba = pd.DataFrame([{
    "comunidad_autonoma": "Madrid. Comunidad de",
    "edad": 20,
    "genero": "hombre",
    "actividad_fisica": "nivel_bajo",
    "asistencia_cine": "si",
    "asistencia_directos": "no_puede",
    "asistencia_cultural": "no_puede",
    "asistencia_deporte": "no_puede",
    "estudios": "educacion_superior",
    "estado_civil": "soltero/a",
    "horasTrabajadas_mes": 130,
    "salario_anual": 10000,
    "satisf_hospitales": "muy_insatisfecho/a",
    "satisf_dentistas": "muy_insatisfecho/a",
    "satisf_especialistas": "muy_insatisfecho/a",
    "satisf_medGeneral": "muy_insatisfecho/a"
}])

In [ ]:
# Prueba de carga mean_df_by_comunidad_loaded y búsqueda de valores
# medios de "dias_alta_contaminacion" y "percepcion_seguridad"
mean_df_by_comunidad_loaded = pd.read_csv(f"{data_basepath}mean_df_by_comunidad.csv")
data_prueba = data_prueba.merge(mean_df_by_comunidad, on="comunidad_autonoma", how="left")
display(data_prueba)

,comunidad_autonoma,edad,genero,actividad_fisica,asistencia_cine,asistencia_directos,asistencia_cultural,asistencia_deporte,estudios,estado_civil,horasTrabajadas_mes,salario_anual,satisf_hospitales,satisf_dentistas,satisf_especialistas,satisf_medGeneral,dias_alta_contaminacion,percepcion_seguridad
0,Madrid. Comunidad de,20,hombre,nivel_bajo,si,no_puede,no_puede,no_puede,educacion_superior,soltero/a,130,10000,muy_insatisfecho/a,muy_insatisfecho/a,muy_insatisfecho/a,muy_insatisfecho/a,6.441595,6.694932


In [ ]:
# Preprocesar y predecir con datos de ejemplo
# data_prueba_prep = preprocessor.transform(data_prueba)
data_prueba_prep = preprocessor_loaded.transform(data_prueba)
prediccion = model_loaded.predict(data_prueba_prep)

print(f"Predicción de nivel_felicidad: {prediccion[0][0]:.2f}")

1/1 [==============================] - 0s 102ms/step
Predicción de nivel_felicidad: 5.45


In [ ]:
import tensorflow as tf
import os

# --- Configuración de path local y Rutas (de tu Celda 3 en Colab) ---

# ASEGÚRATE de que estas rutas son CORRECTAS en tu entorno de Colab.

# LOCAL_BASE_PATH es la ruta base donde tienes tus archivos en Colab/Drive.

LOCAL_BASE_PATH = "/content/sample_data" # O la ruta que uses en Colab

DIRECTORIO_MODELOS_PERSISTENTE = os.path.join(LOCAL_BASE_PATH, 'models')

# Ruta para el modelo .keras que ya funciona en tu Colab

RUTA_MODELO_FINAL = os.path.join(DIRECTORIO_MODELOS_PERSISTENTE, 'modelo_felicidad.keras')

# --- NUEVA RUTA para el SavedModel ---

# El SavedModel se guardará en una CARPETA, no en un archivo .keras

RUTA_SAVED_MODEL_HAPPINESS = os.path.join(DIRECTORIO_MODELOS_PERSISTENTE, 'happiness_level_saved_model')

print(f"Ruta del modelo Keras a cargar en Colab: {RUTA_MODELO_FINAL}")

print(f"Ruta donde se guardará el SavedModel en Colab: {RUTA_SAVED_MODEL_HAPPINESS}")

# 1. Cargar tu modelo Keras existente (como lo haces en la Celda 8 de Colab)

try:

    happiness_calculator_inf = tf.keras.models.load_model(RUTA_MODELO_FINAL)

    print("Predictor de índice de felicidad cargado exitosamente en Colab (formato .keras).")

except Exception as e:

    print(f"ERROR al cargar clasificador de emociones en Colab (formato .keras): {e}")

    print("Por favor, asegúrate de que RUTA_MODELO_FINAL apunta al archivo correcto y que el modelo es válido en Colab.")

    happiness_calculator_inf = None # Asegurarse de que sea None si falla la carga

# 2. Guardar el modelo en formato SavedModel

if happiness_calculator_inf is not None:

    try:

        # Crea la carpeta si no existe

        os.makedirs(RUTA_SAVED_MODEL_HAPPINESS, exist_ok=True)

        # Guarda el modelo en formato SavedModel

        tf.saved_model.save(happiness_calculator_inf, RUTA_SAVED_MODEL_HAPPINESS)

        print(f"Modelo de felicidad guardado exitosamente como SavedModel en: {RUTA_SAVED_MODEL_HAPPINESS}")

    except Exception as e:

        print(f"ERROR al guardar el modelo como SavedModel en Colab: {e}")

else:

    print("No se pudo guardar el modelo como SavedModel porque no se cargó correctamente el modelo .keras inicial.")

Ruta del modelo Keras a cargar en Colab: /content/sample_data/models/modelo_felicidad.keras
Ruta donde se guardará el SavedModel en Colab: /content/sample_data/models/happiness_level_saved_model
Predictor de índice de felicidad cargado exitosamente en Colab (formato .keras).
Modelo de felicidad guardado exitosamente como SavedModel en: /content/sample_data/models/happiness_level_saved_model


In [ ]:
import os

# Ruta de la carpeta que quieres comprimir
# ASEGÚRATE de que esta ruta es CORRECTA y coincide con donde se guardó tu SavedModel
# Según tus logs anteriores, esta debería ser la ruta correcta:
saved_model_folder_path = '/content/sample_data/models/happiness_level_saved_model'

# Nombre del archivo ZIP de salida
zip_file_name = 'happiness_level_saved_model.zip'

# Ruta completa donde se guardará el archivo ZIP de salida
# Lo guardaremos en la misma carpeta padre que el SavedModel
zip_file_path = os.path.join(os.path.dirname(saved_model_folder_path), zip_file_name)
print(f"Comprimiendo la carpeta: {saved_model_folder_path} en {zip_file_path}")

# Comando de terminal para comprimir la carpeta
# El -r es para recursivo (incluir subcarpetas y sus contenidos)
# El {saved_model_folder_path} es la carpeta que se va a comprimir
# El {zip_file_path} es el nombre y la ruta del archivo ZIP resultante
!zip -r {zip_file_path} {saved_model_folder_path}

Comprimiendo la carpeta: /content/sample_data/models/happiness_level_saved_model en /content/sample_data/models/happiness_level_saved_model.zip
  adding: content/sample_data/models/happiness_level_saved_model/ (stored 0%)
  adding: content/sample_data/models/happiness_level_saved_model/assets/ (stored 0%)
  adding: content/sample_data/models/happiness_level_saved_model/variables/ (stored 0%)
  adding: content/sample_data/models/happiness_level_saved_model/variables/variables.index (deflated 59%)
  adding: content/sample_data/models/happiness_level_saved_model/variables/variables.data-00000-of-00001 (deflated 10%)
  adding: content/sample_data/models/happiness_level_saved_model/saved_model.pb (deflated 87%)
  adding: content/sample_data/models/happiness_level_saved_model/fingerprint.pb (stored 0%)
